In [40]:
library(tidyverse)
library(BiocParallel)
library(parallel)
library(sva)
library(dplyr)
library(ggplot2)
library(magrittr)

In [41]:
#command line arguments
args = commandArgs(trailingOnly = TRUE)


In [50]:
#command line arguments for the python script
#countspath = args[1]
countspath =  "./rna_data/counts/rna_counts2.tsv"
#coldatapath = args[2]
coldatapath = "./rna_data/coldata/rna_coldata2.tsv"
#user_directory = args[3]
user_directory = ".rna_data/temp_user/"

In [51]:
#importing the sample data and counts
counts <- readr::read_tsv(countspath)
metadata <- readr::read_tsv(coldatapath) 

Rows: 20242 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr  (1): Hugo_Symbol
dbl (11): Entrez_Gene_Id, GTEX-ZLWG-1026-SM-4WWC4, GTEX-13OVJ-2326-SM-5IJGA,...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 10 Columns: 4
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (3): sample_name, condition, data_source
dbl (1): batch

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [52]:
#creating correct data structures for batch correction
entrez <- counts$Entrez_Gene_Id
#counts$Entrez_Gene_Id <- NULL


df = subset(counts, select = -c(Entrez_Gene_Id) )
rma_expr <- as.matrix(df)

#rma_expr

metadata

rownames(rma_expr) <- counts$Hugo_Symbol
model_m <- model.matrix(~ condition, data = metadata)
batch <- metadata$batch

sample_name,condition,data_source,batch
<chr>,<chr>,<chr>,<dbl>
GTEX-ZLWG-1026-SM-4WWC4,healthy,GTEx,1
GTEX-13OVJ-2326-SM-5IJGA,healthy,GTEx,1
GTEX-14BIM-2226-SM-5SI8Y,healthy,GTEx,1
GTEX-Q734-1626-SM-48U1B,healthy,GTEx,1
GTEX-S7SF-0826-SM-4AD4W,healthy,GTEx,1
GTEX-12WSD-2826-SM-59HKT,healthy,GTEx,2
GTEX-145ME-1326-SM-5O98Q,healthy,GTEx,2
GTEX-S341-1026-SM-4AD71,endometriosis,GTEx,2
GTEX-P4PP-2026-SM-3P61N,healthy,GTEx,2


In [53]:
#getting label for output
labels <- counts$Hugo_Symbol
label <- data.frame(labels)

In [60]:
dim(rma_expr[-1])

NULL

In [71]:
#batch correction!
#bc_rma_expr <- ComBat_seq(rma_expr, batch = batch, mod = model_m, ref.batch = 1)



rma_expr <- rma_expr[,colnames(rma_expr)!="Hugo_Symbol"]

rma_expr

bc_rma_expr <- ComBat_seq(rma_expr, batch = batch, group = NULL)

#bc_rma_expr <- ComBat_seq(rma_expr <- rma_expr[,colnames(rma_expr)!="Hugo_Symbol"], batch = batch)

,GTEX-ZLWG-1026-SM-4WWC4,GTEX-13OVJ-2326-SM-5IJGA,GTEX-14BIM-2226-SM-5SI8Y,GTEX-Q734-1626-SM-48U1B,GTEX-S7SF-0826-SM-4AD4W,GTEX-12WSD-2826-SM-59HKT,GTEX-145ME-1326-SM-5O98Q,GTEX-S341-1026-SM-4AD71,GTEX-P4PP-2026-SM-3P61N,GTEX-113JC-2226-SM-5EGJG
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
FAM208A,2715.29,2192.72,3086.56,2888.33,2074.97,3125.08,2388.48,1791.53,3661.58,2231.58
RADIL,517.00,914.00,884.00,361.00,326.00,708.00,235.00,226.00,1614.00,811.00
AP1M2,4.00,2.00,2.00,6.00,19.00,4.00,5.00,23.00,66.00,3.00
TAF1,2673.00,3581.00,2568.00,2741.00,1711.85,3010.35,2272.00,1632.98,5170.20,2064.30
SIGLEC5,9.49,18.69,11.83,17.94,47.18,3.90,9.26,0.00,6.37,9.34
KLF1,1.00,7.00,1.00,0.00,0.00,0.00,2.00,0.00,0.00,1.00
USHBP1,334.00,529.00,920.62,924.00,1120.00,730.00,457.00,705.00,528.00,205.00
LRRC38,2.00,0.00,2.00,0.00,3.00,0.00,0.00,11.00,2.00,3.00
NKPD1,10.00,9.00,4.76,22.40,6.32,10.00,12.00,6.00,23.00,7.12


ERROR: Error in DGEList(counts = counts): The count matrix is a data.frame instead of a matrix and 10 columns are non-numeric.
  Should these columns be gene annotation instead of counts?


In [ ]:
#editing and combining files
final <- data.frame(bc_rma_expr)
final2 <- bind_cols(label,final)

In [ ]:
#writing out
write_tsv(final2,paste0(user_directory,"bc_counts.tsv"))